## Predicción de dirección diaria (sube/baja) en acciones NASDAQ-100

- Objetivo:

Construir un modelo de clasificación binaria que prediga si el precio de cierre del día siguiente subirá respecto al cierre de hoy, usando únicamente datos históricos de mercado e indicadores técnicos calculados con información pasada.

- Estrategia:

Definición del target: Target = 1 si Close(t+1) > Close(t), por ticker.

Split temporal (sin fuga): entrenamiento con fechas < 2022-01-01, test con fechas ≥ 2022-01-01.

Pipeline completo (preprocesado + modelo) y guardado final en src/models/.

## 1. Contexto del problema

El objetivo del proyecto es construir un modelo de **Machine Learning capaz de predecir la dirección diaria del precio de acciones del NASDAQ-100**.

Se plantea un problema de **clasificación binaria** donde:

- **Target = 1** → el precio de cierre del día siguiente es mayor que el actual
- **Target = 0** → el precio de cierre del día siguiente es menor o igual

El dataset contiene información histórica de mercado como precios (Open, High, Low, Close) y volumen negociado.

A partir de estas variables se generarán **indicadores técnicos** para intentar capturar patrones temporales en la evolución del precio.

In [3]:
import os
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = "src/data_sample/NASDAQ100_Historical_Data.csv"
SPLIT_DATE = pd.Timestamp("2022-01-01")

MODELS_DIR = "src/models"
MODEL_PATH = os.path.join(MODELS_DIR, "finished_pipeline.joblib")

TRAIN_SAMPLES_PATH = os.path.join(MODELS_DIR, "train_samples.csv")
TEST_SAMPLES_PATH = os.path.join(MODELS_DIR, "test_samples.csv")
RESULTS_PATH = os.path.join(MODELS_DIR, "results.csv")

## Paso 1: Carga de datos y checks básicos

In [4]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"No encuentro el archivo en: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

if "Date" not in df.columns or "Ticker" not in df.columns:
    raise ValueError("El dataset debe contener columnas 'Date' y 'Ticker'.")

required_cols = {"Open", "High", "Low", "Close", "Adj Close", "Volume"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Faltan columnas esperadas: {missing}")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Rango fechas:", df["Date"].min().date(), "→", df["Date"].max().date())
print("Tickers:", df["Ticker"].nunique())
df.head() 

Shape: (514075, 8)
Rango fechas: 2000-01-03 → 2026-02-18
Tickers: 100


,Ticker,Date,Open,High,Low,Close,Adj Close,Volume
0,AAPL,2000-01-03,0.94,1.00,0.91,1.00,0.84,535796800
1,AAPL,2000-01-04,0.97,0.99,0.90,0.92,0.77,512377600
2,AAPL,2000-01-05,0.93,0.99,0.92,0.93,0.78,778321600
3,AAPL,2000-01-06,0.95,0.96,0.85,0.85,0.71,767972800
4,AAPL,2000-01-07,0.86,0.90,0.85,0.89,0.75,460734400


## Paso 2: Definición de la variable objetivo

Target = 1 si Close(t+1) > Close(t) dentro del mismo ticker.

In [5]:
df["NextClose"] = df.groupby("Ticker")["Close"].shift(-1)
df = df.dropna(subset=["NextClose"]).copy()
df["Target"] = (df["NextClose"] > df["Close"]).astype(int)
df = df.drop(columns=["NextClose"])

df["Target"].value_counts(normalize=True)

Target
1    0.503238
0    0.496762
Name: proportion, dtype: float64

## Paso 3: Feature engineering por ticker

Se generan indicadores usando solo información presente y pasada dentro de cada ticker:

- returns y log-returns

- lags

- medias móviles

- volatilidad

- momentum

- RSI

- MACD

In [6]:
def add_features_per_ticker(g):
    g = g.sort_values("Date").copy()

    close = g["Close"]
    ret = close.pct_change()
    g["return"] = ret
    g["log_return"] = np.log(close / close.shift(1))

    g["Close_lag1"] = close.shift(1)
    g["Close_lag2"] = close.shift(2)
    g["Close_lag3"] = close.shift(3)

    g["return_lag1"] = ret.shift(1)
    g["return_lag2"] = ret.shift(2)
    g["return_lag3"] = ret.shift(3)

    g["SMA_5"] = close.rolling(5).mean()
    g["SMA_10"] = close.rolling(10).mean()
    g["SMA_20"] = close.rolling(20).mean()

    g["volatility_5"] = ret.rolling(5).std()
    g["volatility_10"] = ret.rolling(10).std()
    g["volatility_20"] = ret.rolling(20).std()

    g["momentum_5"] = close - close.shift(5)
    g["momentum_10"] = close - close.shift(10)

    window = 14
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta.where(delta < 0, 0.0))

    avg_gain = gain.rolling(window=window).mean()
    avg_loss = loss.rolling(window=window).mean()
    rs = avg_gain / avg_loss
    g["RSI"] = 100 - (100 / (1 + rs))

    g["EMA_12"] = close.ewm(span=12, adjust=False).mean()
    g["EMA_26"] = close.ewm(span=26, adjust=False).mean()
    g["MACD"] = g["EMA_12"] - g["EMA_26"]
    g["MACD_signal"] = g["MACD"].ewm(span=9, adjust=False).mean()

    return g

df_fe = df.groupby("Ticker", group_keys=False).apply(add_features_per_ticker)
df_fe = df_fe.dropna().reset_index(drop=True)

print("Shape tras feature engineering:", df_fe.shape)
df_fe.head()

Shape tras feature engineering: (511438, 30)


,Ticker,Date,Open,High,Low,Close,Adj Close,Volume,Target,return,...,volatility_5,volatility_10,volatility_20,momentum_5,momentum_10,RSI,EMA_12,EMA_26,MACD,MACD_signal
0,AAPL,2000-02-01,0.93,0.94,0.89,0.90,0.75,318035200,0,-0.032258,...,0.035202,0.042441,0.052382,-0.10,-0.03,56.603774,0.938587,0.944528,-0.005941,-0.006995
1,AAPL,2000-02-02,0.90,0.91,0.87,0.88,0.74,464195200,1,-0.022222,...,0.035211,0.041947,0.049480,-0.10,-0.07,60.000000,0.929574,0.939748,-0.010175,-0.007631
2,AAPL,2000-02-03,0.90,0.93,0.90,0.92,0.77,475193600,1,0.045455,...,0.046102,0.038930,0.050512,-0.06,-0.09,56.521739,0.928101,0.938285,-0.010185,-0.008142
3,AAPL,2000-02-04,0.93,0.98,0.93,0.96,0.81,425320000,1,0.043478,...,0.036528,0.041935,0.046993,0.05,-0.03,56.521739,0.933008,0.939894,-0.006886,-0.007890
4,AAPL,2000-02-07,0.96,1.02,0.95,1.02,0.85,441067200,1,0.062500,...,0.043351,0.044109,0.047803,0.09,0.07,59.183673,0.946392,0.945828,0.000564,-0.006200


## Paso 4: Split temporal train/test 

Entrenamiento: fechas < 2022-01-01

Test: fechas ≥ 2022-01-01

In [7]:
train_df = df_fe[df_fe["Date"] < SPLIT_DATE].copy()
test_df = df_fe[df_fe["Date"] >= SPLIT_DATE].copy()

print("Train:", train_df.shape, "Test:", test_df.shape)
print("Train rango:", train_df["Date"].min().date(), "→", train_df["Date"].max().date())
print("Test  rango:", test_df["Date"].min().date(), "→", test_df["Date"].max().date())

os.makedirs(MODELS_DIR, exist_ok=True)
train_df[["Ticker", "Date"]].to_csv(TRAIN_SAMPLES_PATH, index=False)
test_df[["Ticker", "Date"]].to_csv(TEST_SAMPLES_PATH, index=False)

TRAIN_SAMPLES_PATH, TEST_SAMPLES_PATH

Train: (408775, 30) Test: (102663, 30)
Train rango: 2000-02-01 → 2021-12-31
Test  rango: 2022-01-03 → 2026-02-17


('src/models\\train_samples.csv', 'src/models\\test_samples.csv')

## Paso 5: Preparación de matrices 

Se eliminan identificadores (Ticker, Date) y se mantiene Target como etiqueta.

In [8]:
drop_cols = ["Ticker", "Date", "Target"]
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X_train = train_df[feature_cols]
y_train = train_df["Target"].astype(int)

X_test = test_df[feature_cols]
y_test = test_df["Target"].astype(int)

print("N features:", len(feature_cols))
X_train.shape, X_test.shape

N features: 27


((408775, 27), (102663, 27))

## Paso 6: Definición de modelos y pipelines

Se entrena un baseline y dos modelos no lineales. Todos usan el mismo preprocesado:

- imputación de nulos

- escalado estándar

In [9]:
preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

pipelines = {
    "LogisticRegression": Pipeline([
        ("prep", preprocess),
        ("model", LogisticRegression(max_iter=2000, n_jobs=-1, random_state=RANDOM_STATE))
    ]),
    "RandomForest": Pipeline([
        ("prep", preprocess),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            min_samples_split=50,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocess),
        ("model", xgb.XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}

list(pipelines.keys())

['LogisticRegression', 'RandomForest', 'XGBoost']

## Paso 7: Entrenamiento y evaluación

Métricas:

- Accuracy

- Precision

- Recall

- F1

- ROC-AUC 

In [10]:
def eval_metrics(y_true, y_pred, y_proba=None):
    out = {}
    out["accuracy"] = accuracy_score(y_true, y_pred)
    out["precision"] = precision_score(y_true, y_pred, zero_division=0)
    out["recall"] = recall_score(y_true, y_pred, zero_division=0)
    out["f1"] = f1_score(y_true, y_pred, zero_division=0)
    out["auc"] = roc_auc_score(y_true, y_proba) if y_proba is not None else np.nan
    return out

rows = []
fitted = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted[name] = pipe

    y_pred = pipe.predict(X_test)
    y_proba = None
    if hasattr(pipe[-1], "predict_proba"):
        y_proba = pipe.predict_proba(X_test)[:, 1]

    m = eval_metrics(y_test, y_pred, y_proba)
    m["model"] = name
    rows.append(m)

resultados_df = pd.DataFrame(rows).sort_values(["auc", "accuracy"], ascending=False)
resultados_df

,accuracy,precision,recall,f1,auc,model
2,0.510125,0.514167,0.780080,0.619806,0.508580,XGBoost
1,0.510505,0.513449,0.834732,0.635808,0.507418,RandomForest
0,0.512434,0.515758,0.777264,0.620067,0.506436,LogisticRegression
